# Appendix A — PyTorch Diagnostic Field Guide

**Book alignment:** PyTorch From First Principles, Appendix A

**Question this notebook isolates:** Do the three most executable field recipes — (1) training-loop separation, (2) attention four-level check, (3) regression paired comparison — each return a falsifiable verdict on a tiny CPU workload?

Coverage: recipes 1 (training loop), 10 (attention), and 14 (regressions) executed below; recipes 2–9 and 11–13 are routed via the symptom table in the closing cell, not executed here.

In [ ]:
import hashlib
import math
import statistics
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Recipe 1: separate forward, backward, and update

Field rule: `delta ≈ -lr * grad` under controlled plain SGD; a healthy gradient on an unowned parameter still yields delta zero.

In [ ]:
torch.manual_seed(0)
p = torch.nn.Parameter(torch.randn(4))
sgd = torch.optim.SGD([p], lr=0.1, momentum=0.0, weight_decay=0.0)
before = p.detach().clone()
loss = (p ** 2).sum()
loss.backward()
g = p.grad.detach().clone()
sgd.step()
delta = p.detach() - before
print('max|delta + lr*g|:', float((delta + 0.1 * g).abs().max()))

lin = nn.Linear(4, 2)
opt = torch.optim.Adam(lin.parameters(), lr=1e-2)
lin_new = nn.Linear(4, 2)
owned = {id(q) for g_ in opt.param_groups for q in g_['params']}
missing_new = [id(q) not in owned for q in lin_new.parameters()]
print('new head owned:', [not m for m in missing_new])

In [ ]:
assert float((delta + 0.1 * g).abs().max()) < 1e-6
assert all(missing_new)
print('recipe 1 verified')

## 2 — Recipe 10: attention four levels in four lines

Shape, head round-trip (semantics), row sums plus forbidden weights (numerics), future-token intervention (behavior).

In [ ]:
B, T, E, Nh = 2, 5, 12, 3
Dh = E // Nh
x = torch.arange(B * T * E).reshape(B, T, E)
split = x.reshape(B, T, Nh, Dh).transpose(1, 2)
rt = split.transpose(1, 2).reshape(B, T, E)
print('L1 shape:', tuple(split.shape), 'L2 round trip:', bool(torch.equal(rt, x)))

torch.manual_seed(1)
q = torch.randn(B, Nh, T, Dh)
k = torch.randn(B, Nh, T, Dh)
v = torch.randn(B, Nh, T, Dh)
allowed = torch.ones(T, T, dtype=torch.bool).tril()
scores = q @ k.transpose(-2, -1) / math.sqrt(Dh)
w = torch.softmax(scores.masked_fill(~allowed, float('-inf')), dim=-1)
out = w @ v
ref = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
print('L3 row err:', float((w.sum(-1) - 1).abs().max()), 'forbidden:', float(w[..., ~allowed].abs().max()))
print('L4 vs SDPA:', float((out - ref).abs().max()))

with torch.no_grad():
    i = 2
    x1 = torch.randn(B, T, E)
    x2 = x1.clone()
    x2[:, i + 1:] = torch.randn(B, T - i - 1, E)
    proj = nn.Linear(E, E, bias=False)
    def run(z):
        h = proj(z).reshape(B, T, Nh, Dh).transpose(1, 2)
        return F.scaled_dot_product_attention(h, h, h, is_causal=True).transpose(1, 2).reshape(B, T, E)
    d_pre = float((run(x1)[:, :i + 1] - run(x2)[:, :i + 1]).abs().max())
    d_suf = float((run(x1)[:, i + 1:] - run(x2)[:, i + 1:]).abs().max())
print(f'L4 intervention prefix={d_pre:.2e} suffix={d_suf:.2e}')

In [ ]:
assert tuple(split.shape) == (2, 3, 5, 4)
assert torch.equal(rt, x)
assert float((w.sum(-1) - 1).abs().max()) < 1e-5
assert float(w[..., ~allowed].abs().max()) < 1e-6
assert float((out - ref).abs().max()) < 1e-5
assert d_pre < 1e-6 and d_suf > 1e-5
print('recipe 10 verified')

## 3 — Recipe 14: baseline spread plus paired deltas plus fingerprint

Same seed repeats exactly; six seeds set the spread; a 10x-LR candidate shifts every paired delta one way; identical weights under a scaled eval path move the metric while the fingerprint changes.

In [ ]:
g = torch.Generator().manual_seed(0)
W0 = torch.randn(32, 4, generator=g)
Xtr = torch.randn(600, 32, generator=g)
Xva = torch.randn(200, 32, generator=torch.Generator().manual_seed(5))
ytr_clean = (Xtr @ W0).argmax(-1)
yva = (Xva @ W0).argmax(-1)
ytr = ytr_clean.clone()
flip = torch.rand(600, generator=g) < 0.30
ytr[flip] = torch.randint(0, 4, (int(flip.sum()),), generator=g)

def run(seed, lr=0.02):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 4))
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=3e-3)
    m.train()
    for ep in range(8):
        perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(seed + ep))
        for i in range(0, len(Xtr), 128):
            idx = perm[i:i + 128]
            o.zero_grad()
            F.cross_entropy(m(Xtr[idx]), ytr[idx]).backward()
            o.step()
    m.eval()
    with torch.no_grad():
        return float(F.cross_entropy(m(Xva), yva))

r0a, r0b = run(0), run(0)
base = [run(s) for s in range(6)]
cand = [run(s, lr=0.2) for s in range(6)]
deltas = [c - b for b, c in zip(base, cand)]
sd = statistics.pstdev(base)
shift = statistics.median(cand) - statistics.median(base)
print(f'repeat {r0a:.4f}=={r0b:.4f}; spread={max(base) - min(base):.4f} shift={shift:.4f} ({shift / sd:.1f} sd)')

torch.manual_seed(0)
m = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 4))
m.eval()
with torch.no_grad():
    lc = float(F.cross_entropy(m(Xva), yva))
    ls = float(F.cross_entropy(m(Xva * 1.15), yva))
fp = lambda s: hashlib.sha256(s.encode()).hexdigest()[:8]
print(f'weights fixed: canonical={lc:.4f} scaled={ls:.4f} fp same={fp("a") == fp("b")}')
print('eval fp differ:', fp('eval|1.0') != fp('eval|1.15'))

In [ ]:
assert r0a == r0b
assert all(d > 0 for d in deltas)
assert shift > 3 * sd
assert abs(ls - lc) > 1e-4
print('recipe 14 verified')

## What we earned

Three packets, three verdicts: the update equals `-lr*grad` under controlled SGD and unowned heads never move; attention needs shape, round-trip, row sums, and intervention — no single level suffices; a regression needs spread, paired deltas, and an eval fingerprint before any mechanism story.

Routing for the rest: tensor trace (ch 2) for far-downstream shape errors; autograd reachability (ch 3) for missing grads; four-structure audit (ch 5) for vanishing layers; producer/consumer split (ch 6) for starved GPUs; stage trace (ch 7) for wrong representations; shape ledger (ch 8) for CNN geometry; feature inspector (ch 9) for misread scores; learning ledger (ch 11) for flat learning; measurement contract (ch 12) for slow code; guard evidence (ch 13) for compile surprises.